# UBER- Eats Order Tracking Partner Performance Evaluation

In [2]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [4]:
df_orders = pd.read_csv('../Data/016/fct_orders.csv', parse_dates=['expected_delivery_time','actual_delivery_time','order_date'])

pl_orders = pl.read_csv('../Data/016/fct_orders.csv', try_parse_dates=True)

# Pregunta 1

### ¿Cuál es el porcentaje de pedidos entregados a tiempo en enero de 2024? Considera que un pedido está a tiempo si su actual_delivery_time es menor o igual a su expected_delivery_time. Esto nos ayudará a evaluar la precisión general del seguimiento.

```SQL
SELECT
    ROUND(COUNT((CASE WHEN (actual_delivery_time <= expected_delivery_time) THEN 1 END))* 100.0 / COUNT(*),2)
FROM fct_orders
WHERE ((EXTRACT(MONTH FROM order_date) = 1) AND
       (EXTRACT(YEAR FROM order_date) = 2024));
```

In [9]:
df_january = df_orders[
    (df_orders['order_date'].dt.month == 1) &
    (df_orders['order_date'].dt.year == 2024)
]

on_time_pct = (
    (df_january['actual_delivery_time'] <= df_january['expected_delivery_time'])
    .mean() * 100
).round(2)

In [10]:
res = pl_orders.filter(
    (pl.col('order_date').dt.month() == 1) &
    (pl.col('order_date').dt.year() == 2024)
).select(
    on_time_percentage = (
        (pl.col('actual_delivery_time') <= pl.col('expected_delivery_time'))
        .mean() * 100
    ).round(2)
)

# Pregunta 2

### Haz una lista de los 5 mejores repartidores en enero de 2024, clasificados por el mayor porcentaje de entregas a tiempo. Utiliza el campo delivery_partner_name de los registros. Esto nos ayudará a identificar qué socios tienen el mejor desempeño.

```SQL
SELECT
    delivery_partner_name,
    ROUND(COUNT((CASE WHEN (actual_delivery_time <= expected_delivery_time) THEN 1 END))* 100.0 / COUNT(*),2) AS porcentage_delivery_success
FROM fct_orders
WHERE ((EXTRACT(MONTH FROM order_date) = 1) AND
       (EXTRACT(YEAR FROM order_date) = 2024))
GROUP BY delivery_partner_name
ORDER BY porcentage_delivery_success DESC
LIMIT 5;
```

In [15]:
df_jan = df_orders[
    (df_orders['order_date'].dt.month == 1) &
    (df_orders['order_date'].dt.year == 2024)
].copy()

df_jan['is_on_time'] = df_jan['actual_delivery_time'] <= df_jan['expected_delivery_time']

top_5_partners = (
    df_jan.groupby('delivery_partner_name')['is_on_time']
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

In [21]:
pl_jan = (pl_orders.filter(
    (pl.col('order_date').dt.month() == 1) &
    (pl.col('order_date').dt.year() == 2024)
).group_by('delivery_partner_name').agg(
    porcentage_delivery_success = (
        (pl.col('actual_delivery_time') <= pl.col('expected_delivery_time'))
        .mean() * 100
    ).round(2)
)
.sort('porcentage_delivery_success', descending=True)
.head(5)
)

# Pregunta 3

### Identifica al repartidor (o repartidores) en enero de 2024 cuyo porcentaje de entregas a tiempo sea inferior al 50%. Devuelve sus nombres de repartidor en mayúsculas (UPPERCASE). Necesitamos trabajar con estos socios repartidores para mejorar sus tasas de entrega a tiempo.

In [34]:
df_jan = df_orders[
    (df_orders['order_date'].dt.month == 1) &
    (df_orders['order_date'].dt.year == 2024)
].copy()

partners_performance = (
    df_jan.assign(on_time = df_jan['actual_delivery_time'] <= df_jan['expected_delivery_time'])
    .groupby('delivery_partner_name')['on_time']
    .mean() * 100
)

res_pandas = (
    partners_performance[partners_performance < 50]
    .index.str.upper()
    .tolist()
)

res_pandas

['DAWN']

In [40]:
res_polars = (
    pl_orders
    .filter(
        (pl.col('order_date').dt.month() == 1) &
        (pl.col('order_date').dt.year() == 2024)
    )
    .group_by('delivery_partner_name')
    .agg(
        success_rate = (
            (pl.col('actual_delivery_time') <= pl.col('expected_delivery_time'))
            .mean() * 100
        )
    )
    .filter(pl.col('success_rate') < 50)
    .select(
        pl.col('delivery_partner_name').str.to_uppercase()
    )
)

res_polars

delivery_partner_name
str
"""DAWN"""
